# Responsible AI Analysis — KC House Price Prediction

**Dataset**: King County House Sales (21,613 transactions) + Census demographics  
**Baseline Model**: RobustScaler + KNeighborsRegressor (provided)  
**Improved Model**: GradientBoostingRegressor with full feature set  
**Frameworks**: Microsoft ErrorAnalysis + ResponsibleAI + SHAP  

This notebook addresses the challenge requirements:
- **Evaluate** the provided model — accuracy, overfitting/underfitting
- **Improve** — more features, better algorithm, proper train/test evaluation
- **Explain** — SHAP interpretability, error analysis, counterfactual what-if, causal effects

## Dashboard Components
1. **Model Overview** — Performance metrics, predicted vs actual, residuals
2. **Error Analysis** — Microsoft ErrorAnalysis tree & heatmap, cohort breakdowns
3. **Model Interpretability** — Global/local SHAP feature importance
4. **Counterfactual What-If** — Minimal changes to shift predictions
5. **Causal Analysis** — Treatment effects of actionable features on price

## 0. Setup

In [ ]:
# Dependencies (already in conda_environment.yml)
# !pip install responsibleai raiwidgets erroranalysis interpret shap scikit-learn pandas numpy matplotlib seaborn econml --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Setup complete')

## 1. Load Data & Model

In [ ]:
# Load raw data (same sources as create_model.py)
sales = pd.read_csv('../data/kc_house_data.csv', dtype={'zipcode': str})
demographics = pd.read_csv('../data/zipcode_demographics.csv', dtype={'zipcode': str})

# --- Baseline model: same 7 features as create_model.py ---
BASELINE_COLS = ['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
                 'floors', 'sqft_above', 'sqft_basement', 'zipcode']
baseline_data = sales[BASELINE_COLS].merge(demographics, how='left', on='zipcode').drop(columns='zipcode')

y_baseline = baseline_data.pop('price')
X_baseline = baseline_data

# --- Improved model: all useful features + demographics ---
IMPROVED_COLS = ['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
                 'floors', 'waterfront', 'view', 'condition', 'grade',
                 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated',
                 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
improved_data = sales[IMPROVED_COLS].merge(demographics, how='left', on='zipcode').drop(columns='zipcode')

y_full = improved_data.pop('price')
X_full = improved_data

# Same random split for fair comparison
X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_baseline, y_baseline, test_size=0.2, random_state=42)
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42)

print(f'Dataset: {len(sales):,} home sales')
print(f'Baseline features: {X_baseline.shape[1]} | Improved features: {X_full.shape[1]}')
print(f'Train: {len(X_base_train):,} | Test: {len(X_base_test):,}')
print(f'Target range: ${y_baseline.min():,.0f} — ${y_baseline.max():,.0f}')

In [ ]:
# Train both models
# Baseline: exact same pipeline as create_model.py
baseline_model = make_pipeline(
    RobustScaler(),
    KNeighborsRegressor()
).fit(X_base_train, y_base_train)

# Improved: GradientBoosting with all features
improved_model = GradientBoostingRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42
).fit(X_full_train, y_full_train)

print(f'Baseline: {type(baseline_model[-1]).__name__} ({X_base_train.shape[1]} features)')
print(f'Improved: {type(improved_model).__name__} ({X_full_train.shape[1]} features)')

---
## 2. Model Overview — Baseline vs Improved

Compare the provided KNN model against our improved GradientBoosting model.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Return train/test metrics for a model."""
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    def calc_metrics(y_true, y_pred):
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R²': r2_score(y_true, y_pred),
            'MAPE (%)': mean_absolute_percentage_error(y_true, y_pred) * 100,
            'Median AE': np.median(np.abs(y_true - y_pred))
        }
    
    return calc_metrics(y_train, y_pred_train), calc_metrics(y_test, y_pred_test)

base_train_m, base_test_m = evaluate_model(baseline_model, X_base_train, X_base_test, y_base_train, y_base_test)
impr_train_m, impr_test_m = evaluate_model(improved_model, X_full_train, X_full_test, y_full_train, y_full_test)

comparison = pd.DataFrame({
    'Metric': list(base_train_m.keys()),
    'Baseline Train': [f'{v:,.2f}' for v in base_train_m.values()],
    'Baseline Test': [f'{v:,.2f}' for v in base_test_m.values()],
    'Improved Train': [f'{v:,.2f}' for v in impr_train_m.values()],
    'Improved Test': [f'{v:,.2f}' for v in impr_test_m.values()],
})

print('Model Comparison')
print('=' * 90)
print(f'Baseline: KNN ({X_base_train.shape[1]} features) | Improved: GradientBoosting ({X_full_train.shape[1]} features)')
print()
comparison

In [ ]:
# Predicted vs Actual — side-by-side comparison
y_pred_base = baseline_model.predict(X_base_test)
y_pred_impr = improved_model.predict(X_full_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_true, y_pred, title in [
    (axes[0], y_base_test, y_pred_base, 'Baseline (KNN)'),
    (axes[1], y_full_test, y_pred_impr, 'Improved (GradientBoosting)')
]:
    ax.scatter(y_true, y_pred, alpha=0.3, s=10, c='steelblue')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    ax.text(0.05, 0.92, f'R² = {r2:.3f}\nMAE = ${mae:,.0f}',
            transform=ax.transAxes, fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('Actual Price ($)')
    ax.set_ylabel('Predicted Price ($)')
    ax.set_title(f'{title} — Predicted vs Actual')
    ax.legend()

plt.tight_layout()
plt.savefig('01_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual distribution — improved model
residuals = y_full_test - y_pred_impr

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', lw=2)
axes[0].axvline(residuals.median(), color='orange', linestyle='--', lw=2, label=f'Median: ${residuals.median():,.0f}')
axes[0].set_xlabel('Residual ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distribution (Test — Improved Model)')
axes[0].legend()

axes[1].scatter(y_pred_impr, residuals, alpha=0.3, s=10, c='steelblue')
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residuals vs Predicted (Test — Improved Model)')

plt.tight_layout()
plt.savefig('02_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Residual stats: mean=${residuals.mean():,.0f}, std=${residuals.std():,.0f}, skew={residuals.skew():.2f}')

---
## 3. Error Analysis — Microsoft ErrorAnalysis + Cohort Breakdown

Using [erroranalysis](https://erroranalysis.ai/) — the same engine that powers the Azure ML Responsible AI Dashboard's Error Analysis component.

In [ ]:
from erroranalysis import ModelAnalyzer, report

# Prepare test data with target column for ErrorAnalysis
CATEGORICAL_FEATURES = ['waterfront', 'view', 'condition', 'grade', 'floors']
TARGET = 'price'

test_with_target = X_full_test.copy()
test_with_target[TARGET] = y_full_test.values

# Create the ModelAnalyzer — same engine as Azure RAI Error Analysis tab
analyzer = ModelAnalyzer(
    model=improved_model,
    dataset=test_with_target,
    true_y=y_full_test.values,
    feature_names=X_full_test.columns.tolist(),
    categorical_features=CATEGORICAL_FEATURES,
    target_column=TARGET
)

# Generate error report
error_report = report.ModelAnalysisReport(analyzer)
print('ErrorAnalysis ModelAnalyzer created')
print(f'Features: {len(X_full_test.columns)}')
print(f'Categorical: {CATEGORICAL_FEATURES}')
print(f'Test samples: {len(X_full_test)}')

In [ ]:
# Error tree from ErrorAnalysis — identifies high-error cohorts automatically
from erroranalysis._internal.error_analyzer import ErrorAnalyzer

ea = ErrorAnalyzer(
    model=improved_model,
    dataset=test_with_target,
    true_y=y_full_test.values,
    feature_names=X_full_test.columns.tolist(),
    categorical_features=CATEGORICAL_FEATURES,
    target_column=TARGET
)

# Get the error tree structure
tree_info = ea.compute_error_tree(
    features=X_full_test.columns.tolist(),
    max_depth=3,
    num_leaves=31,
    min_child_samples=20
)

print('Error Tree computed by Microsoft ErrorAnalysis')
print(f'Tree nodes: {len(tree_info)}')

# Also build a visual surrogate tree for static plots
from sklearn.tree import DecisionTreeRegressor, plot_tree

errors = np.abs(y_full_test.values - y_pred_impr)
error_tree = DecisionTreeRegressor(max_depth=3, max_leaf_nodes=31, min_samples_leaf=20)
error_tree.fit(X_full_test, errors)

fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    error_tree,
    feature_names=X_full_test.columns.tolist(),
    filled=True, rounded=True, ax=ax, fontsize=9,
    impurity=False, proportion=True
)
ax.set_title('Error Tree — Identifying High-Error Cohorts (Surrogate)', fontsize=16, fontweight='bold')
plt.savefig('03_error_tree.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Error heatmap: sqft_living × grade (top error-correlated features)
error_df = X_full_test.copy()
error_df['abs_error'] = errors
error_df['sqft_living_bin'] = pd.qcut(error_df['sqft_living'], q=8, duplicates='drop')
error_df['grade_bin'] = pd.Categorical(error_df['grade'])

heatmap_data = error_df.pivot_table(
    values='abs_error', index='grade_bin', columns='sqft_living_bin', aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    heatmap_data, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax,
    cbar_kws={'label': 'Mean Absolute Error ($)'}
)
ax.set_xlabel('sqft_living (binned)', fontsize=13)
ax.set_ylabel('grade', fontsize=13)
ax.set_title('Error Heatmap — sqft_living × grade', fontsize=16, fontweight='bold')
plt.savefig('04_error_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cohort analysis — error by segments
cohorts = {
    'waterfront': error_df.groupby('waterfront')['abs_error'].agg(['mean', 'count']),
    'grade': error_df.groupby('grade')['abs_error'].agg(['mean', 'count']),
    'condition': error_df.groupby('condition')['abs_error'].agg(['mean', 'count']),
    'view': error_df.groupby('view')['abs_error'].agg(['mean', 'count']),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (name, cohort_df) in zip(axes.flatten(), cohorts.items()):
    bars = ax.bar(cohort_df.index.astype(str), cohort_df['mean'], color='steelblue', alpha=0.8)
    for bar, count in zip(bars, cohort_df['count']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f'n={count}', ha='center', fontsize=9)
    ax.set_xlabel(name)
    ax.set_ylabel('Mean Absolute Error ($)')
    ax.set_title(f'Error by {name}')

plt.suptitle('Error Analysis by Cohort', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('05_error_cohorts.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Model Interpretability — Feature Importance (SHAP)

In [ ]:
# SHAP TreeExplainer for GradientBoosting (native, fast)
explainer = shap.TreeExplainer(improved_model)
shap_values = explainer.shap_values(X_full_test)
feature_names = X_full_test.columns.tolist()

print(f'SHAP values shape: {shap_values.shape}')
print(f'Base value (E[f(x)]): ${explainer.expected_value:,.0f}')

In [ ]:
# Global feature importance — mean |SHAP|
shap.summary_plot(
    shap_values, X_full_test,
    feature_names=feature_names, plot_type='bar',
    max_display=20, show=False
)
plt.title('Global Feature Importance (mean |SHAP|)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('06_shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm — shows direction of feature impact
shap.summary_plot(
    shap_values, X_full_test,
    feature_names=feature_names, max_display=20, show=False
)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('07_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP dependence plots for top features
top_features = ['sqft_living', 'grade', 'lat', 'yr_built']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feat in zip(axes.flatten(), top_features):
    feat_idx = feature_names.index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, X_full_test,
        feature_names=feature_names, ax=ax, show=False
    )

plt.suptitle('SHAP Dependence Plots — Top Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('08_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Local explanation — Individual predictions (waterfall plots)
idx_cheap = y_full_test.idxmin()
idx_expensive = y_full_test.idxmax()
idx_worst = pd.Series(np.abs(y_full_test.values - y_pred_impr), index=y_full_test.index).idxmax()

for label, idx in [('Cheapest home', idx_cheap), ('Most expensive', idx_expensive), ('Highest error', idx_worst)]:
    row_pos = list(X_full_test.index).index(idx)
    actual = y_full_test.loc[idx]
    predicted = y_pred_impr[row_pos]
    
    print(f'\n--- {label} ---')
    print(f'  Actual: ${actual:,.0f} | Predicted: ${predicted:,.0f} | Error: ${abs(actual - predicted):,.0f}')
    
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[row_pos],
            base_values=explainer.expected_value,
            data=X_full_test.iloc[row_pos],
            feature_names=feature_names
        ),
        max_display=15, show=False
    )
    plt.title(f'Local Explanation — {label}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'09_local_{label.replace(" ", "_").lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 5. Counterfactual What-If Analysis

In [ ]:
from responsibleai import RAIInsights

# Build full train/test DataFrames with target for RAIInsights
train_df = X_full_train.copy()
train_df[TARGET] = y_full_train.values
test_df = X_full_test.copy()
test_df[TARGET] = y_full_test.values

# RAIInsights — counterfactual analysis
rai = RAIInsights(
    model=improved_model,
    train=train_df,
    test=test_df.head(500),  # subsample for speed
    target_column=TARGET,
    task_type='regression',
    categorical_features=CATEGORICAL_FEATURES
)

# Counterfactual: what minimal changes move price into desired range?
rai.counterfactual.add(
    total_CFs=10,
    desired_range=[75000, 7700000]
)

print('Computing counterfactuals...')
rai.counterfactual.compute()
print('Done!')

In [ ]:
# Inspect counterfactual results
cf_results = rai.counterfactual.get()

if cf_results:
    cf_obj = cf_results[0]
    print(f'Counterfactual analysis results available')
    print(f'Number of test points analyzed: {len(cf_obj.cf_examples_list)}')
    
    # Show example: first test point's counterfactuals
    example_idx = 0
    cf_example = cf_obj.cf_examples_list[example_idx]
    
    print(f'\n--- Original datapoint ---')
    print(cf_example.test_instance_df.to_string())
    
    print(f'\n--- Counterfactual alternatives ---')
    print(cf_example.final_cfs_df.to_string())
else:
    print('No counterfactual results generated')

In [ ]:
# Manual what-if analysis — easy to explain in presentation
# Pick a median-priced house and show effect of changes
median_idx = (y_full_test - y_full_test.median()).abs().idxmin()
baseline = X_full_test.loc[[median_idx]].copy()
baseline_price = improved_model.predict(baseline)[0]

print(f'Baseline house: ${baseline_price:,.0f}')
print(f'sqft_living: {baseline["sqft_living"].values[0]}, grade: {baseline["grade"].values[0]}, bathrooms: {baseline["bathrooms"].values[0]}\n')

scenarios = []
for feat, changes in [
    ('sqft_living', [+200, +500, +1000]),
    ('grade', [+1, +2]),
    ('bathrooms', [+0.5, +1]),
    ('waterfront', [+1]),
    ('view', [+1, +2]),
]:
    for delta in changes:
        modified = baseline.copy()
        modified[feat] = modified[feat] + delta
        new_price = improved_model.predict(modified)[0]
        scenarios.append({
            'Feature': feat,
            'Change': f'+{delta}',
            'New Value': modified[feat].values[0],
            'New Price': f'${new_price:,.0f}',
            'Price Delta': f'${new_price - baseline_price:+,.0f}',
            'Price Delta %': f'{(new_price - baseline_price) / baseline_price * 100:+.1f}%'
        })

what_if_df = pd.DataFrame(scenarios)
print('What-If Scenarios — Changes to a Median-Priced Home')
what_if_df

---
## 6. Causal Analysis — Treatment Effects

In [ ]:
from econml.dml import LinearDML
from sklearn.linear_model import LassoCV

# Causal inference using EconML (same backend as Azure RAI causal component)
TREATMENT_FEATURES = ['sqft_living', 'grade', 'bathrooms']
causal_results = {}

for treatment_feat in TREATMENT_FEATURES:
    print(f'\n=== Causal Analysis: {treatment_feat} → price ===')
    
    T = train_df[[treatment_feat]]
    Y = train_df[TARGET]
    X_confounders = train_df.drop(columns=[TARGET, treatment_feat])
    
    dml = LinearDML(
        model_y=LassoCV(cv=3),
        model_t=LassoCV(cv=3),
        random_state=42
    )
    dml.fit(Y, T, X=X_confounders)
    
    ate = dml.ate(X=X_confounders)
    ate_inference = dml.ate_inference(X=X_confounders)
    
    ci_lower = ate_inference.conf_int_mean()[0][0]
    ci_upper = ate_inference.conf_int_mean()[1][0]
    
    print(f'  ATE: ${ate[0]:,.2f} per unit increase')
    print(f'  95% CI: [${ci_lower:,.2f}, ${ci_upper:,.2f}]')
    print(f'  P-value: {ate_inference.pvalue()[0]:.4f}')
    
    causal_results[treatment_feat] = {
        'ate': ate[0], 'ci_lower': ci_lower, 'ci_upper': ci_upper,
        'pvalue': ate_inference.pvalue()[0], 'model': dml
    }

In [ ]:
# Visualize causal effects
fig, ax = plt.subplots(figsize=(10, 6))

features = list(causal_results.keys())
ates = [causal_results[f]['ate'] for f in features]
ci_lowers = [causal_results[f]['ci_lower'] for f in features]
ci_uppers = [causal_results[f]['ci_upper'] for f in features]
errors = [[ate - ci_l for ate, ci_l in zip(ates, ci_lowers)],
          [ci_u - ate for ate, ci_u in zip(ates, ci_uppers)]]

bars = ax.barh(features, ates, xerr=errors, color='steelblue', alpha=0.8, capsize=5)
ax.axvline(0, color='red', linestyle='--', lw=1)
ax.set_xlabel('Average Treatment Effect on Price ($)')
ax.set_title('Causal Effects — Per Unit Increase in Treatment Feature', fontsize=14, fontweight='bold')

for i, (feat, ate) in enumerate(zip(features, ates)):
    ax.text(ate, i, f'  ${ate:,.0f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('10_causal_effects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heterogeneous treatment effects — does sqft_living impact vary by segment?
sqft_model = causal_results['sqft_living']['model']
X_confounders_test = test_df.drop(columns=[TARGET, 'sqft_living'])

ite = sqft_model.effect(X=X_confounders_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(ite, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(ite.mean(), color='red', linestyle='--', lw=2, label=f'Mean: ${ite.mean():,.0f}')
axes[0].set_xlabel('Individual Treatment Effect ($/sqft)')
axes[0].set_ylabel('Count')
axes[0].set_title('Heterogeneous Effects: sqft_living → price')
axes[0].legend()

ite_df = X_full_test[['waterfront', 'lat']].copy()
ite_df['effect'] = ite.flatten()

waterfront_effects = ite_df.groupby('waterfront')['effect'].agg(['mean', 'std', 'count'])
axes[1].bar(
    ['No Waterfront', 'Waterfront'],
    waterfront_effects['mean'],
    yerr=waterfront_effects['std'] / np.sqrt(waterfront_effects['count']),
    color=['steelblue', 'coral'], alpha=0.8, capsize=5
)
axes[1].set_ylabel('Treatment Effect ($/sqft)')
axes[1].set_title('Effect of sqft_living by Waterfront Status')

for i, (_, row) in enumerate(waterfront_effects.iterrows()):
    axes[1].text(i, row['mean'], f'${row["mean"]:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Heterogeneous Causal Effects', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('11_heterogeneous_effects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Geographic heterogeneity — effect of sqft_living by latitude
fig, ax = plt.subplots(figsize=(12, 6))

scatter = ax.scatter(
    X_full_test['lat'], ite.flatten(),
    c=X_full_test['long'], cmap='RdYlBu_r', alpha=0.5, s=15
)
plt.colorbar(scatter, label='Longitude')
ax.set_xlabel('Latitude')
ax.set_ylabel('Treatment Effect: sqft_living → price ($/sqft)')
ax.set_title('Geographic Variation of sqft_living Impact on Price', fontsize=14, fontweight='bold')
ax.axhline(ite.mean(), color='red', linestyle='--', lw=1, alpha=0.5, label=f'Mean: ${ite.mean():,.0f}')
ax.legend()

plt.tight_layout()
plt.savefig('12_geographic_causal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Summary — Causal Effects Table

In [ ]:
summary = []
for feat, res in causal_results.items():
    summary.append({
        'Treatment Feature': feat,
        'ATE ($ per unit)': f"${res['ate']:,.2f}",
        '95% CI Lower': f"${res['ci_lower']:,.2f}",
        '95% CI Upper': f"${res['ci_upper']:,.2f}",
        'P-value': f"{res['pvalue']:.4f}",
        'Significant': '✓' if res['pvalue'] < 0.05 else '✗'
    })

summary_df = pd.DataFrame(summary)
print('\nCausal Treatment Effects Summary')
print('=' * 80)
summary_df

---
## 8. Full Interactive RAI Dashboard

Launches the complete Microsoft Responsible AI Dashboard with all components — Error Analysis, Explainability, Counterfactuals, and Causal Analysis in a single interactive view.

In [ ]:
# Full RAI Dashboard with all components
# NOTE: requires Jupyter with widget support (not plain script execution)
TREATMENT_FEATURES_CAUSAL = ['sqft_living', 'grade', 'bathrooms']
HETEROGENEITY_FEATURES = ['lat', 'waterfront']

rai_full = RAIInsights(
    model=improved_model,
    train=train_df,
    test=test_df.head(1000),
    target_column=TARGET,
    task_type='regression',
    categorical_features=CATEGORICAL_FEATURES
)

# Add all dashboard components
rai_full.explainer.add()
rai_full.error_analysis.add(
    max_depth=3, num_leaves=31, min_child_samples=20,
    filter_features=['sqft_living', 'grade', 'waterfront', 'lat']
)
rai_full.counterfactual.add(total_CFs=10, desired_range=[75000, 7700000])
rai_full.causal.add(
    treatment_features=TREATMENT_FEATURES_CAUSAL,
    heterogeneity_features=HETEROGENEITY_FEATURES,
    nuisance_model='linear',
    heterogeneity_model='linear'
)

print('Computing all RAI components (this may take a few minutes)...')
rai_full.compute()
print('All components computed!')

# Save for reproducibility
rai_full.save('./rai_insights_output')

In [ ]:
# Launch the interactive dashboard (opens in browser)
from raiwidgets import ResponsibleAIDashboard

ResponsibleAIDashboard(rai_full)

---
## Key Findings

### Model Comparison
- **Baseline** (KNN, 7 features + demographics): underfits — missing critical predictors like grade, waterfront, lat/long
- **Improved** (GradientBoosting, 18 features + demographics): significant R² and MAE improvement through proper feature selection and a more expressive algorithm

### Model Interpretability (SHAP)
- **sqft_living** and **grade** are the dominant predictors
- **lat/long** capture neighborhood premium effects (Seattle core vs suburbs)
- Census-derived features (demographics join) provide additional predictive signal

### Error Analysis (ErrorAnalysis + Cohorts)
- Highest errors concentrated in **high-grade + large sqft** segment (luxury homes)
- **Waterfront** properties have disproportionately higher error due to small sample size and high price variance
- The model underpredicts luxury homes — consider segment-specific models

### Causal Insights (EconML)
- **sqft_living**: each additional sqft causally increases price (heterogeneous by geography)
- **grade**: upgrading construction quality has significant causal impact on home value
- **bathrooms**: adding a bathroom has a measurable causal effect on price
- Effects are heterogeneous — waterfront properties show different treatment response

### Actionable Recommendations
- For homeowners: **grade upgrades** and **sqft expansions** are the most cost-effective levers for increasing home value
- For the model: consider segment-specific models for luxury/waterfront properties where errors are highest
- For the API: the improved GradientBoosting model should replace the KNN baseline